# SSM / Satisfaction-of-Search Effect on LWS Probability

Tests whether target-visits are more likely to become LWS once *another* target has already been
identified in the trial (Cain et al., 2013, "A taxonomy of errors in multiple-target visual search"; Adamo
et al., 2013, "Self-induced attentional blink"), and whether that effect depends on the trial's rendering
(`trial_category`: COLOR/BW/NOISE) or the missed target's own category (`target_category`,
`ImageCategoryEnum`, 6 levels).

- **Q1**: is there an SSM/SoS effect at all - `P[LWS | any prior hit]` vs. `P[LWS | no prior hit]`?
- **Q3**: does the effect differ by trial type?
- **Q4**: does the effect differ by the missed target's category?

Shared data prep, `visit_type` classification, and predictor construction live in `ssm.py` (this package's
shared module). Every model below is fit as a flat `(1|subject)` vs. `(1|subject/trial)`-nested pair - see
`CODE_REVIEW.md` M10 for why (the unit of observation is a visit, and visits nest within trial within
subject).

In [ ]:
import os

from analysis.helpers.r_bridge import setup_rpy2, to_r_dataframe, source_r, get_r_object, glmer_metrics, cached_fit
setup_rpy2()

import pytensor
pytensor.config.cxx = ""  # this machine's C++ toolchain can't compile pytensor's C backend - see fit_bambi_nested_m10_audit.py
import bambi as bmb
import arviz as az

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import config as cnfg
from analysis.ssm_and_ab.ssm import load_ssm_funnel, add_ssm_predictors

pio.renderers.default = "notebook"      # "notebook" or "browser"

In [ ]:
data, funnel, hits = load_ssm_funnel()
funnel = add_ssm_predictors(funnel, hits)
print(f"{len(funnel)} valid-trial target-visits, {int(funnel['any_prior_hit'].sum())} with a prior hit in the same trial")
funnel.groupby("any_prior_hit", observed=True)["is_lws"].agg(n="size", lws_rate="mean")

### Descriptive: P[LWS] with vs. without a prior hit

In [ ]:
subject_stats = (
    funnel.groupby(["subject", "any_prior_hit"], observed=True)
    .agg(lws_rate=("is_lws", "mean"))
    .reset_index()
)
pop_stats = (
    subject_stats.groupby("any_prior_hit")
    .agg(mean_rate=("lws_rate", "mean"), sem_rate=("lws_rate", "sem"))
    .reset_index()
)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=["No Prior Hit", "Prior Hit"],
    y=pop_stats["mean_rate"],
    error_y=dict(type="data", array=1.96 * pop_stats["sem_rate"], visible=True),
    marker_color=[cnfg.get_discrete_color(0), cnfg.get_discrete_color(1)],
))
fig.update_layout(
    title="P[LWS | prior hit in trial] (subject-level mean +/- 95% CI)",
    yaxis=dict(title="P[LWS]", range=[0, max(0.3, pop_stats["mean_rate"].max() * 1.4)]),
    template="plotly_white", width=500, height=400,
)
fig.show()

### Q1 (SSM/SoS effect) / Q3 (by trial type) / Q4 (by missed-target category) - frequentist GLMMs

All three questions are one R script (`analysis/R/ssm_effect_glmm.R`): `m1_*` (Q1, `is_lws ~
any_prior_hit`), `m3_*` (Q3, `... * trial_category`), `m4_*` (Q4, `... * target_category`), each as a flat
vs. subject/trial-nested pair.

In [ ]:
def _fit_ssm_effect_glmm():
    model_data = funnel[["subject", "trial", "trial_category", "target_category", "any_prior_hit", "is_lws"]].copy()
    to_r_dataframe(model_data, "dat")
    source_r(os.path.join(os.getcwd(), "..", "R", "ssm_effect_glmm.R"))
    return {
        name: glmer_metrics(get_r_object(name), name)
        for name in ["m1_flat", "m1_nested", "m3_flat", "m3_nested", "m4_flat", "m4_nested"]
    }


ssm_effect_glmm_result = cached_fit(os.path.join(os.getcwd(), "..", "R", "_cache", "ssm_effect_glmm.pkl"), _fit_ssm_effect_glmm)

pd.DataFrame([
    {"model": k, "aic": v["aic"], "bic": v["bic"], "r2_marginal": v["r2_marginal"], "converged": v["converged"]}
    for k, v in ssm_effect_glmm_result.items()
])

In [ ]:
print("Q1 headline (nested model): any_prior_hit coefficient")
ssm_effect_glmm_result["m1_nested"]["coefficients"]

In [ ]:
print("Q3: any_prior_hit x trial_category (nested model)")
ssm_effect_glmm_result["m3_nested"]["coefficients"]

In [ ]:
print("Q4: any_prior_hit x target_category (nested model)")
ssm_effect_glmm_result["m4_nested"]["coefficients"]

### Bayesian companion (Q1)

Same `is_lws ~ any_prior_hit + (1|subject/trial)` specification, fit via `bambi`/`pymc` - no `rpy2`/`brms`
bridge exists for this (see the plan doc). 2000 draws / 1000 tune / 4 chains, matching the draws/tune/chains
convention already established in `stimulus_features.ipynb` / `fit_bambi_nested_m10_audit.py` - `cores=1`
here, though, not that script's `cores=2`: `nbconvert`'s headless execution can't pickle the model for
multiprocessing sampling on Windows (`ParallelSamplingError: The model could not be unpickled`), so chains
run sequentially instead of in parallel.

Only Q1 gets a Bayesian companion here - Q3/Q4's interaction models are cheap to add the same way if the
frequentist interaction terms above turn out interesting enough to warrant it.

In [ ]:
BAMBI_KWARGS = dict(draws=2000, tune=1000, chains=4, cores=1, target_accept=0.95, random_seed=42, progressbar=False)

bayes_cache_path = os.path.join(os.getcwd(), "..", "R", "_cache", "ssm_effect_bayes_m1_idata.nc")
bayes_data = funnel[["subject", "trial", "any_prior_hit", "is_lws"]].copy()
bayes_model = bmb.Model("is_lws ~ any_prior_hit + (1|subject/trial)", bayes_data, family="bernoulli")

if os.path.exists(bayes_cache_path):
    bayes_idata = az.from_netcdf(bayes_cache_path)
    print(f"Loaded cached idata from {bayes_cache_path}")
else:
    bayes_idata = bayes_model.fit(**BAMBI_KWARGS)
    os.makedirs(os.path.dirname(bayes_cache_path), exist_ok=True)
    az.to_netcdf(bayes_idata, bayes_cache_path)
    print(f"Fit complete, saved to {bayes_cache_path}")

az.summary(bayes_idata, var_names=["Intercept", "any_prior_hit"])

In [ ]:
n_div = int(bayes_idata.sample_stats["diverging"].sum())
print(f"Divergences: {n_div}")

posterior_effect = bayes_idata["posterior"]["any_prior_hit"]
print(f"P[effect < 0] = {(posterior_effect < 0).mean().item():.4f}   (negative -> prior hit REDUCES P[LWS])")

az.plot_posterior(bayes_idata, var_names=["any_prior_hit"], hdi_prob=0.95)

### Notes

- Check whether the frequentist `any_prior_hit` coefficient (Q1 cell above) is negative - i.e. a prior hit
  in the trial is associated with *lower* LWS probability, the opposite direction the SSM/SoS account
  predicts. If the Bayesian posterior above agrees, that is a substantive finding worth reporting as such,
  not a bug to chase.
- Q3/Q4 (trial-type and missed-target-category moderation) are printed above from
  `ssm_effect_glmm_result["m3_*"]` / `["m4_*"]`.